# pandas-pilot · LangGraph Workshop

> Build a ReAct agent that analyzes CSVs from scratch, and learn to manage it as an engineered system with trace + eval.

## §0. Opening

By the end of today you will see this conversation:

```
USER: Analyze data/sales.csv. Which region has the lowest average revenue?
AGENT:
  → calls inspect_data("data/sales.csv")
  ← returns shape, columns, missing values, first five rows
  → calls run_pandas("data/sales.csv", "df.dropna(subset=['region','revenue']).groupby('region')['revenue'].mean()")
  ← returns the average revenue of East/West/North/South
AGENT: The region with the lowest average revenue is South, at about XXX. Missing values in region/revenue were excluded.
```

Just follow the instructor and keep running. **Don't only look at the cells. Open up the returned objects and look inside.** That matters even more than listening.


## §1. Environment first aid

### Get a Google AI API key

1. Go to https://aistudio.google.com/apikey and get a free Gemini API key
2. Copy the key and paste it between the quotes in the cell below

> ⚠️ **This is only a workshop shortcut.** In real development, use an environment variable or a Colab secret, and **never commit an API key into a repo**.


Make sure the packages are installed first (the first run on Colab takes a while).

In [4]:
%pip install -q -U langgraph langchain langchain-google-genai pandas

Note: you may need to restart the kernel to use updated packages.


c:\Users\User\Documents\GitHub\langgraph-harness-workshop\.venv\Scripts\python.exe: No module named pip


Paste the **whole** API key you just got between the quotes below (replace the placeholder too, and don't leave any extra text before or after). A Google API key is 39 characters long and starts with `AIza`.

In [ ]:
import os

# Written like this only as a workshop shortcut. Don't do this in a real project
GOOGLE_API_KEY = "PASTE_YOUR_KEY_HERE"

# Safety check: catches a missing / wrong paste, or an accidentally pasted placeholder, right here
assert GOOGLE_API_KEY != "PASTE_YOUR_KEY_HERE", "Paste your key in first"
assert GOOGLE_API_KEY.isascii(), "The key contains non-ASCII characters; you probably left the placeholder text in by accident"
assert GOOGLE_API_KEY.startswith("AIza"), "A Google API key should start with AIza; check that you pasted the right thing"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("API key length:", len(GOOGLE_API_KEY))


API key length: 39


In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)
result = llm.invoke("Who are you? Answer in one sentence")
print(result.content)


I am an AI trained by Google, designed to help you find information, create, and communicate.


## §2. The LangGraph mental model

LangGraph has only three concepts. Remember them and you are halfway there:

- **State**: a dict-like object shared by all nodes. Think of it as a "whiteboard".
- **Node**: a function that reads State and returns the part to update.
- **Edge**: the arrows between nodes that decide who runs next.

A whole graph run is: `State → Node A → update State → Edge decides the next one → Node B → ...`

In the second half of the workshop, an agent is also just a combination of these three things. Don't think of it as magic.


## §3. Python crash course

LangGraph uses type hints heavily, so let's go over them quickly first.


### §3.1 type hints

In [7]:
# Variables
x: int = 20
name: str = "Alice"

# Function parameters and return value
def add(a: int, b: int) -> int:
    return a + b

# The Python runtime does not check hints (they are just annotations); an IDE / type checker does
print(add(1, 2))


3


### §3.2 TypedDict: a structured dict

In [8]:
from typing import TypedDict


class User(TypedDict):
    name: str
    age: int


u: User = {"name": "Alice", "age": 30}
print(u)
print(type(u).__name__)  # Note that it is still a dict underneath


{'name': 'Alice', 'age': 30}
dict


### §3.3 Annotated: attach a label to a type

In [9]:
from typing import Annotated

# Annotated[original type, "description / handling rule"]
Score = Annotated[int, "an integer between 0 and 100"]

s: Score = 87
print(s)

# LangGraph uses Annotated to tell the graph "how to merge this field" (we will use it again later)


87


### §3.4 The `@tool` decorator

In [10]:
from langchain_core.tools import tool


@tool
def add_two(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


# A tool object can be called directly, but more importantly it has a schema for the LLM to read
print(add_two.invoke({"a": 3, "b": 4}))
print("---")
print(add_two.name)
print(add_two.description)
print(add_two.args)


7
---
add_two
Add two integers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## §4. The first graph: an echo bot

The smallest runnable graph. It does one thing: return the user's message unchanged.


In [11]:
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict):
    text: str


def echo(state: State) -> dict:
    # Note: return a dict whose keys are the state fields to update
    return {"text": state["text"]}


builder = StateGraph(State)
builder.add_node("echo", echo)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
graph = builder.compile()

result = graph.invoke({"text": "Hello, LangGraph"})
print(result)


{'text': 'Hello, LangGraph'}


c:\Users\User\Documents\GitHub\langgraph-harness-workshop\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Open up the result object and take a look.

In [12]:
print(type(result).__name__)
print(repr(result))
print(result["text"])


dict
{'text': 'Hello, LangGraph'}
Hello, LangGraph


## §5. Two nodes + an edge (clean → respond)

Connect two nodes: clean first, then respond. Learn how nodes hand off to each other.


In [13]:
class State2(TypedDict):
    text: str


def clean(state: State2) -> dict:
    # Strip the message and lowercase it
    return {"text": state["text"].strip().lower()}


def respond(state: State2) -> dict:
    return {"text": f"I heard you say: {state['text']}"}


b = StateGraph(State2)
b.add_node("clean", clean)
b.add_node("respond", respond)
b.add_edge(START, "clean")
b.add_edge("clean", "respond")
b.add_edge("respond", END)
g2 = b.compile()

print(g2.invoke({"text": "  HELLO World  "}))


{'text': 'I heard you say: hello world'}


## §6. Dissecting the LLM object

Next we will connect the LLM to a graph. First, let's look closely at what the LLM object itself returns.


### §6.1 llm.invoke

In [14]:
resp = llm.invoke("Answer in three words: what is Python?")
print(resp)


content='Programming language' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019e0eac-2af1-7501-8849-f26d8b266a7b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 10, 'output_tokens': 503, 'total_tokens': 513, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 501}}


### §6.2 Open up the whole AIMessage

In [15]:
print("type:", type(resp).__name__)
print("content:", repr(resp.content))
print("tool_calls:", resp.tool_calls)
print("usage_metadata:", resp.usage_metadata)
print("response_metadata keys:", list(resp.response_metadata.keys()))


type: AIMessage
content: 'Programming language'
tool_calls: []
usage_metadata: {'input_tokens': 10, 'output_tokens': 503, 'total_tokens': 513, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 501}}
response_metadata keys: ['finish_reason', 'model_name', 'safety_ratings', 'model_provider']


Note that in newer Gemini versions (2.5+ with reasoning), `resp.content` returns a **list of dicts**
containing `type='text'` and a thought signature. To get a plain text string, use the `.text` property instead:


In [16]:
# .content is raw; it may be a str or a list of dicts
print("raw content:", repr(resp.content)[:200], "...")

# .text is a uniform plain-string accessor
print("text:", resp.text)


raw content: 'Programming language' ...
text: Programming language


**Rule of thumb: use `.text` to show to people, and use `.content` for message serialization or when you need the full structure.**


### §6.3 Multi-turn conversation: messages is a list, the LLM has no memory

In [17]:
from langchain_core.messages import AIMessage, HumanMessage

# The LLM itself has no memory; you must stuff the whole conversation history into it
history = [
    HumanMessage(content="My name is Alice"),
    AIMessage(content="Hi Alice!"),
    HumanMessage(content="What is my name?"),
]
print(llm.invoke(history).text)


Your name is Alice.


In [18]:
# Counter-example: do not pass the history, only the last sentence
print(llm.invoke("What is my name?").text)


I don't know your name.

I'm an AI and can't get or remember a user's personal information, so I don't know what your name is.

If you tell me, I can call you by whatever name you like during this conversation.


## §7. MessagesState

LangGraph has a built-in `MessagesState` for the common case of "accumulating a message list".


In [19]:
from langgraph.graph import MessagesState


def chatbot(state: MessagesState) -> dict:
    response = llm.invoke(state["messages"])
    # Note: return a list; LangGraph appends it automatically
    return {"messages": [response]}


b = StateGraph(MessagesState)
b.add_node("chatbot", chatbot)
b.add_edge(START, "chatbot")
b.add_edge("chatbot", END)
g3 = b.compile()

result = g3.invoke({"messages": [HumanMessage(content="Hi, my name is Alice")]})
for m in result["messages"]:
    print(type(m).__name__, "-", m.content)


HumanMessage - Hi, my name is Alice
AIMessage - Hello Alice, nice to meet you!

I am a large language model trained by Google. How can I help you?


### Counter-example: using a plain dict instead of MessagesState

If you don't use MessagesState and use a plain list with no reducer, new messages **overwrite** old ones instead of being appended. MessagesState internally uses `Annotated[list, add_messages]`, which tells the graph "merge this field with add_messages".


## §8. Tool Calling (the core of the first half)

Connect tools to the LLM so it can call the functions we write.


### §8.1 What the LLM returns after bind_tools

In [20]:
from langchain_core.tools import tool


@tool
def get_weather(city: str) -> str:
    """Look up the weather of a city."""
    return f"{city} is sunny, 25 degrees"


llm_with_tools = llm.bind_tools([get_weather])
resp = llm_with_tools.invoke("What is the weather like in Taipei today?")
print("content:", repr(resp.content))
print("tool_calls:", resp.tool_calls)


content: ''
tool_calls: [{'name': 'get_weather', 'args': {'city': 'Taipei'}, 'id': '032ae5d1-330d-430f-8791-d34d56fc3ef9', 'type': 'tool_call'}]


Notice two things:

1. **content is an empty string.** The LLM decided to call a tool first and has not produced the final answer yet
2. **tool_calls is a list of dicts**, and each dict has name / args / id

The LLM does not run the tool automatically. It only "instructs that it be called". Running it is your job (ToolNode will do it for you later).


### §8.2 The docstring is the LLM's only "user manual"

In [21]:
# The LLM cannot see your code. It only sees the tool's name, docstring, and arguments
print(get_weather.name)
print(get_weather.description)  # this is the docstring
print(get_weather.args)


get_weather
Look up the weather of a city.
{'city': {'title': 'City', 'type': 'string'}}


**A badly written docstring → nobody uses the tool.** Soon we will write an example where the docstring forces the LLM to "inspect first, then run".


### §8.3 Two tools: inspect_data / run_pandas

In [22]:
import io

import pandas as pd
from langchain_core.tools import tool


@tool
def inspect_data(path: str) -> str:
    """Check the quality of a CSV. You **must** call this tool before running
    any groupby/filter/agg, to learn the data's shape, columns, and missing values.

    Args:
        path: path to the CSV file
    Returns:
        a string containing the shape, columns, missing values, and the first five rows
    """
    df = pd.read_csv(path)
    buf = io.StringIO()
    buf.write(f"shape: {df.shape}\n")
    buf.write(f"columns: {list(df.columns)}\n")
    buf.write("\ndtypes:\n")
    buf.write(df.dtypes.to_string())
    buf.write("\n\nmissing values per column:\n")
    buf.write(df.isna().sum().to_string())
    buf.write("\n\nhead(5):\n")
    buf.write(df.head(5).to_string())
    return buf.getvalue()


@tool
def run_pandas(path: str, code: str) -> str:
    """Run pandas analysis code on a CSV, such as groupby, filter, agg, or sort.
    The loaded DataFrame is provided as the variable `df` and can be used directly in code.

    Args:
        path: path to the CSV file
        code: a pandas expression (for example "df.groupby('region')['revenue'].mean()")
    Returns:
        the result as a string (truncated beyond 2000 characters)
    """
    # ⚠️ Note: eval() is a serious security problem in production
    # It is simplified here for the workshop. In practice, use a sandbox / RestrictedPython / e2b
    df = pd.read_csv(path)
    try:
        result = eval(code, {"df": df, "pd": pd})
    except Exception as exc:
        return f"Execution error: {type(exc).__name__}: {exc}"
    text = str(result)
    if len(text) > 2000:
        text = text[:2000] + "\n... (output truncated)"
    return text


tools = [inspect_data, run_pandas]
print([t.name for t in tools])


['inspect_data', 'run_pandas']


Prepare demo data: if `data/sales.csv` does not exist, generate one on the spot. It deliberately mixes in 5 region NaNs + 5 revenue NaNs so inspect_data can see them.

In [23]:
import os
import random
from datetime import date, timedelta

CSV_PATH = "data/sales.csv"

if not os.path.exists(CSV_PATH):
    os.makedirs(os.path.dirname(CSV_PATH), exist_ok=True)
    rng = random.Random(42)
    regions = ["East", "West", "North", "South"]
    products = ["Widget", "Gadget", "Gizmo", "Doohickey"]
    base = date(2025, 1, 1)
    rows = []
    for _ in range(100):
        region = rng.choice(regions)
        unit_price = rng.uniform(8, 18) if region == "South" else rng.uniform(20, 60)
        units = rng.randint(1, 50)
        rows.append({
            "date": (base + timedelta(days=rng.randint(0, 89))).isoformat(),
            "region": region,
            "product": rng.choice(products),
            "units": units,
            "revenue": round(units * unit_price, 2),
        })
    df = pd.DataFrame(rows)
    df.loc[rng.sample(range(100), 5), "region"] = pd.NA
    df.loc[rng.sample(range(100), 5), "revenue"] = pd.NA
    df.to_csv(CSV_PATH, index=False)
    print(f"Generated {CSV_PATH} ({len(df)} rows)")
else:
    print(f"{CSV_PATH} already exists")


Generated data/sales.csv (100 rows)


First test that the tool runs by itself.

In [24]:
print(inspect_data.invoke({"path": CSV_PATH}))


shape: (100, 5)
columns: ['date', 'region', 'product', 'units', 'revenue']

dtypes:
date           str
region         str
product        str
units        int64
revenue    float64

missing values per column:
date       0
region     5
product    0
units      0
revenue    5

head(5):
         date region    product  units  revenue
0  2025-02-01   East     Gadget     18   378.01
1  2025-03-11   West     Widget     44  2176.19
2  2025-01-28  South     Gadget      6    49.91
3  2025-03-25   East  Doohickey     46  1952.69
4  2025-01-01   West     Gadget     18   683.43


### §8.4 Assemble the graph: ToolNode + tools_condition

In [25]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

llm_with_tools = llm.bind_tools(tools)


def chatbot(state: MessagesState) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("chatbot", chatbot)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "chatbot")
# tools_condition: if the last message has tool_calls, go to the tools node; otherwise go to END
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")  # after tools run, loop back to chatbot

agent = builder.compile()
print("agent compiled.")


agent compiled.


### §8.5 Run the same question three times

In [37]:
question = f"Analyze {CSV_PATH}. Which region has the lowest average revenue?"

for i in range(3):
    print(f"========== Run {i + 1} ==========")
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    # Use .text to get a plain string (see §6.2 for why)
    print(result["messages"][-1].text)
    print()


========== Run 1 ==========
The South region has the lowest average revenue.

========== Run 2 ==========
The region with the lowest average revenue is South.

========== Run 3 ==========
The region with the lowest average revenue is South.



**This is the nature of being stochastic.** Three inconsistent answers are perfectly normal; sometimes it even runs without inspecting first. In the second half we will manage it with trace + eval.


### §8.6 Open up result['messages']

In [27]:
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

for i, m in enumerate(result["messages"]):
    print(f"--- [{i}] {type(m).__name__} ---")
    tc = getattr(m, "tool_calls", None)
    if tc:
        for call in tc:
            print(f"  tool_call -> {call['name']}({call['args']})")
    else:
        # .text gets a plain string from both content shapes, str and list-of-dict
        text = m.text if hasattr(m, "text") else str(m.content)
        if len(text) > 300:
            text = text[:300] + "..."
        print(f"  {text}")


--- [0] HumanMessage ---
  Analyze data/sales.csv. Which region has the lowest average revenue?
--- [1] AIMessage ---
  tool_call -> inspect_data({'path': 'data/sales.csv'})
--- [2] ToolMessage ---
  shape: (100, 5)
columns: ['date', 'region', 'product', 'units', 'revenue']

dtypes:
date           str
region         str
product        str
units        int64
revenue    float64

missing values per column:
date       0
region     5
product    0
units      0
revenue    5

head(5):
         date regi...
--- [3] AIMessage ---
  tool_call -> run_pandas({'path': 'data/sales.csv', 'code': "df.dropna(subset=['region', 'revenue']).groupby('region')['revenue'].mean().idxmin()"})
--- [4] ToolMessage ---
  South
--- [5] AIMessage ---
  The region with the lowest average revenue is South.


This is what the ReAct loop really looks like:

```
HumanMessage → AIMessage(tool_calls=[inspect_data]) → ToolMessage → AIMessage(tool_calls=[run_pandas]) → ToolMessage → AIMessage(final answer)
```

On every round the LLM sees "all the messages so far", so it can see all the earlier tool results.


## §9. First-half recap

So far you have:

- Seen the three things of LangGraph: State / Node / Edge
- Written your own echo / clean+respond / chatbot
- Opened up the LLM object and know what content / tool_calls / usage_metadata each are
- Assembled your first tool-using agent with `bind_tools` + `ToolNode` + `tools_condition`
- **Seen with your own eyes the same question getting three different answers**

But two problems are still unsolved:

1. **The three runs differ.** How do you know which one is "right"?
2. **You can't tell whether changing the prompt helped.** Going by gut feeling?

The second half deals with these two things.


## §10. Opening the second half: what is a harness

Harness = treat the agent as a "black-box device under test" and wrap tooling around it, so you can:

1. **Observe** what it did (trace)
2. **Score** how well it did (eval)
3. **Go back, change it,** and run again (loop)

Next, §11 is trace, §12 is eval, and §13 puts them together into a loop.


## §11. Trace

`graph.invoke` only gives you the final result, and you can't see the decisions in between.
Switch to `graph.stream(..., stream_mode="updates")` and you can see every step.


In [28]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="updates",
):
    print(event)
    print("---")


{'chatbot': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'inspect_data', 'arguments': '{"path": "data/sales.csv"}'}, '__gemini_function_call_thought_signatures__': {'e1f0b216-a9b4-4e8f-b8a4-3364989df229': 'CsMEAQw51se7n93DCLColqQmQ+abDXOYjuwtPrwW18Jh8ehCwzoWpK4TDHxlR6s25TrQYXF542YdYoh84YaJwfXYYorEDz/l8WApSs+cnUNPwub5jdCoyqf94Yhb+4u/QfyYCBhXjbjKkiZ9qGhTv9WV8ZCayvH62qXmlE6bUCYqOwatCY5rFMS7QwiZfNz7tcsEpsrKHYXM3SbmSF/8hvYqk1zmYLNh0ysg/hwlT4mI8p0BYF2M1QZEJX6Tw4l8Ye/8YblKWIxnKl/sVLIJ9QZeK78pU0eVPwgH+XKVIo7QvMTkZ+slk9PSn1EiQP7PvmE18yL/t0genStB9CJa8qJ4itA2pLCbK0JB1dF9aIIpgslMWJGrjG8+0uUaUBQy6Duf3rlTkTPSZTmkXirbona3KFxMGU5gUYlqwEDSwh7zMW/Djk29GTDsmOFHfjE8c86mmx8DPyiYNjWOKeHiS3jGlbBUNWH927sp73yZ6GDsyyLsnz0zuf+MGAGHftmsrw+jjfDlTzh5WtND3vddulj7xGQW6/j5TfNpJsQoV/XzeQHIo2XxD7Hc2Nv2v3l/uGM0DhiqUORzy3DxRMDfx9rK2d7DWYFKpZ75UC7DfKruN4cfNm+nmo7TPHjyukxSs+RyI/vaCefxpP7Y3HMMYXFsbbLO1E4QWDgMjt669KvYCyecuZp7jsY2z3+wQ9pLZ3s42/6+LlWPACbJhlHynep8P/KrmKpK9fOLg2BgtUh6wbMTIDzEoJ

The raw events are too noisy. Write a pretty_trace to list them in a way people can read.

In [29]:
def pretty_trace(graph, user_message: str) -> list[dict]:
    events = []
    print(f"USER: {user_message}\n")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    for step, event in enumerate(graph.stream(inputs, stream_mode="updates"), start=1):
        events.append(event)
        for node_name, node_state in event.items():
            print(f"--- step {step} | node: {node_name} ---")
            for msg in node_state.get("messages", []):
                role = type(msg).__name__
                tool_calls = getattr(msg, "tool_calls", None)
                if tool_calls:
                    for call in tool_calls:
                        print(f"  [{role}] tool_call -> {call['name']}({call['args']})")
                else:
                    text = msg.text if hasattr(msg, "text") else str(msg.content)
                    if len(text) > 400:
                        text = text[:400] + "..."
                    print(f"  [{role}] {text}")
            print()
    return events


events = pretty_trace(agent, question)


USER: Analyze data/sales.csv. Which region has the lowest average revenue?

--- step 1 | node: chatbot ---
  [AIMessage] tool_call -> inspect_data({'path': 'data/sales.csv'})

--- step 2 | node: tools ---
  [ToolMessage] shape: (100, 5)
columns: ['date', 'region', 'product', 'units', 'revenue']

dtypes:
date           str
region         str
product        str
units        int64
revenue    float64

missing values per column:
date       0
region     5
product    0
units      0
revenue    5

head(5):
         date region    product  units  revenue
0  2025-02-01   East     Gadget     18   378.01
1  2025-03-11   West  ...

--- step 3 | node: chatbot ---
  [AIMessage] tool_call -> run_pandas({'path': 'data/sales.csv', 'code': "df.dropna(subset=['region', 'revenue']).groupby('region')['revenue'].mean().sort_values().head(1)"})

--- step 4 | node: tools ---
  [ToolMessage] region
South    382.347727
Name: revenue, dtype: float64

--- step 5 | node: chatbot ---
  [AIMessage] The region with the

## §12. Eval

Trace lets you "see". Eval lets you "score". Let's write 5 cases first.


In [30]:
EVAL_CASES = [
    {
        "id": "001_must_inspect_first",
        "user_message": f"Give me an overview of the data in {CSV_PATH}.",
        "must_call_first": "inspect_data",
        "must_contain_in_final": [],
    },
    {
        "id": "002_handle_empty_result",
        "user_message": f"In {CSV_PATH}, count the records where region is 'Mars'.",
        "must_call_first": "inspect_data",
        "must_contain_in_final": ["0"],
    },
    {
        "id": "003_inspect_before_summary",
        "user_message": f"Calculate the average revenue in {CSV_PATH}.",
        "must_call_first": "inspect_data",
        "must_contain_in_final": [],
    },
    {
        "id": "004_top_region",
        "user_message": f"Analyze {CSV_PATH}. Which region has the lowest average revenue?",
        "must_call_first": "inspect_data",
        "must_contain_in_final": [],
    },
    {
        "id": "005_count_rows",
        "user_message": f"How many rows of data are in {CSV_PATH}?",
        "must_call_first": "inspect_data",
        "must_contain_in_final": [],
    },
]
print(f"{len(EVAL_CASES)} cases in total")


5 cases in total


Write two helpers: one to extract the order of tool_calls, one to extract the last piece of text.

In [31]:
def extract_tool_calls(events):
    names = []
    for event in events:
        for _node, state in event.items():
            for msg in state.get("messages", []) or []:
                for call in getattr(msg, "tool_calls", None) or []:
                    names.append(call["name"])
    return names


def extract_final_text(events):
    final = ""
    for event in events:
        for node, state in event.items():
            if node != "chatbot":
                continue
            for msg in state.get("messages", []) or []:
                if not getattr(msg, "tool_calls", None):
                    final = msg.text if hasattr(msg, "text") else str(getattr(msg, "content", "") or "")
    return final


In [32]:
def evaluate_one(graph, case):
    inputs = {"messages": [{"role": "user", "content": case["user_message"]}]}
    events = list(graph.stream(inputs, stream_mode="updates"))
    tc = extract_tool_calls(events)
    final = extract_final_text(events)

    reasons = []
    if case["must_call_first"]:
        if not tc:
            reasons.append(f"No tool was called (expected {case['must_call_first']})")
        elif tc[0] != case["must_call_first"]:
            reasons.append(f"The first tool was {tc[0]}, expected {case['must_call_first']}")
    for kw in case.get("must_contain_in_final", []):
        if kw not in final:
            reasons.append(f"The final reply is missing the keyword {kw!r}")

    return {"id": case["id"], "passed": not reasons, "reasons": reasons,
            "tool_calls": tc, "final_text": final}


def run_eval_suite(graph, cases):
    results = []
    for c in cases:
        r = evaluate_one(graph, c)
        mark = "PASS" if r["passed"] else "FAIL"
        print(f"[{mark}] {r['id']}  tool_calls={r['tool_calls']}")
        for reason in r["reasons"]:
            print(f"        - {reason}")
        results.append(r)
    p = sum(1 for r in results if r["passed"])
    print(f"\nPassed {p}/{len(results)} ({p / len(results):.0%})")
    return results


### baseline: an agent with no system prompt

In [33]:
baseline_results = run_eval_suite(agent, EVAL_CASES)


[PASS] 001_must_inspect_first  tool_calls=['inspect_data']
[FAIL] 002_handle_empty_result  tool_calls=['run_pandas']
        - The first tool was run_pandas, expected inspect_data
[FAIL] 003_inspect_before_summary  tool_calls=['run_pandas']
        - The first tool was run_pandas, expected inspect_data
[FAIL] 004_top_region  tool_calls=['run_pandas']
        - The first tool was run_pandas, expected inspect_data
[PASS] 005_count_rows  tool_calls=['inspect_data']

Passed 2/5 (40%)


You will usually see **40-60% pass**. The agent often skips inspect and runs run_pandas directly.

Next, add a system prompt and run again.


### v2: add a system prompt

In [34]:
from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """You are a careful data analysis assistant.

Workflow rules:
1. After receiving any question, first call inspect_data to see the data's shape, columns, and missing values.
2. Only after inspect_data may you call run_pandas to do the analysis.
3. Handle all missing values before reporting the result.
4. Reply in English, giving the conclusion first and then the key numbers.
"""


def chatbot_v2(state: MessagesState) -> dict:
    msgs = state["messages"]
    if not msgs or not isinstance(msgs[0], SystemMessage):
        msgs = [SystemMessage(content=SYSTEM_PROMPT), *msgs]
    return {"messages": [llm_with_tools.invoke(msgs)]}


b2 = StateGraph(MessagesState)
b2.add_node("chatbot", chatbot_v2)
b2.add_node("tools", ToolNode(tools))
b2.add_edge(START, "chatbot")
b2.add_conditional_edges("chatbot", tools_condition)
b2.add_edge("tools", "chatbot")
agent_v2 = b2.compile()

v2_results = run_eval_suite(agent_v2, EVAL_CASES)


[PASS] 001_must_inspect_first  tool_calls=['inspect_data']
[PASS] 002_handle_empty_result  tool_calls=['inspect_data', 'run_pandas']
[PASS] 003_inspect_before_summary  tool_calls=['inspect_data', 'run_pandas']
[PASS] 004_top_region  tool_calls=['inspect_data', 'run_pandas', 'run_pandas']
[PASS] 005_count_rows  tool_calls=['inspect_data']

Passed 5/5 (100%)


Expect **80-100% pass**.

The key point: **you now have an objective number.** You can say "v2 is better than baseline", "change the prompt again and watch the score", "run it in CI and block regressions". The agent goes from art to engineering.


## §13. The Harness Loop concept

Chain trace + eval together and you have a dev loop:

```
1. Change the agent (prompt / tool / graph)
2. run_eval_suite(agent)
3. See which case dropped → pretty_trace(agent, that_case) → find the problem
4. Go back to 1
```

**This loop is the core of agent engineering.** Not LangGraph, and not Gemini. With any framework and any LLM, as long as you can run this loop, you can maintain it.

Advanced: run the eval in CI, add a coverage matrix (case x model x prompt), add cost / latency metrics. All of these are extensions of this loop.


## §14. Wrap-up + after-class exercises

What you learned today:

- LangGraph: State / Node / Edge / MessagesState / ToolNode / tools_condition
- What tool calling really looks like (the ReAct loop)
- Why agents are stochastic, and why the docstring matters
- Using trace to see the middle steps, and eval to get an objective score
- The harness loop is the foundation of agent engineering

### After-class exercises

1. **Add a new case:** make baseline pass but v2 fail, and observe prompt over-fitting
2. **A third tool:** add `plot_data(path, code)` that produces a base64 PNG, and write a case to check it is not misused
3. **Cross-model comparison:** swap `gemini-2.5-flash` for `gemini-2.5-pro`, run the same EVAL_CASES, and look at the score gap
4. **Improve the trace:** add latency / token usage to pretty_trace and you can see the cost of each case
5. **Put it in CI:** write EVAL_CASES into pytest and run it nightly in GitHub Actions

PRs / issues are welcome: https://github.com/AluminumShark/langgraph-harness-workshop
